# 01 — Data preparation (UA-DETRAC → YOLO)

Runtime: **CPU** is enough (no GPU needed, it saves your GPU quota).

Raw data is downloaded to the Colab disk (`/content/raw`), never to the repo. Only the converted dataset (1 frame out of 10) is saved to Google Drive.

⏸️ = checkpoint: send the output to your mentor before going on.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/vehicle-counting/data'
!mkdir -p {DRIVE_DIR}
!df -h /content | tail -1

## 2. Clone the repo and install

In [ ]:
%cd /content
!rm -rf vehicle-counting
!git clone -q https://github.com/louchenilounes/vehicle-counting.git
%cd /content/vehicle-counting
!pip install -q -e ".[dev]"
!pytest -q

## 3. Download UA-DETRAC (official Google Drive links from the authors)

Source: https://sites.google.com/view/daweidu/projects/ua-detrac

If gdown fails with *"Too many users have viewed or downloaded this file"*: open the link in your browser, click **Add shortcut to Drive**, then use the fallback cell below.

In [ ]:
!pip install -q -U gdown
!mkdir -p /content/raw
%cd /content/raw
# Annotations (folder)
!gdown --folder https://drive.google.com/drive/folders/1am_x3ywLJnABq4REtzbi0XTu5csFrp8v -O annotations
# Images (one big file)
!gdown 1fQC6CEWOeL9pnJ4ZdD7U2gromcnWDRgm -O images_archive
%cd /content/vehicle-counting

In [ ]:
# FALLBACK only if gdown failed: copy from the shortcut in your Drive
# !cp -r "/content/drive/MyDrive/<shortcut name>" /content/raw/

## 4. Explore and extract ⏸️

Send the output of the two cells below to your mentor before going on.

In [ ]:
!ls -lhR /content/raw | head -60
!du -sh /content/raw

In [ ]:
import shutil, zipfile, tarfile
from pathlib import Path

RAW = Path('/content/raw')
EXTRACT = RAW / 'extracted'
EXTRACT.mkdir(exist_ok=True)

for f in sorted(p for p in RAW.rglob('*') if p.is_file() and EXTRACT not in p.parents):
    if zipfile.is_zipfile(f) or tarfile.is_tarfile(f):
        print('Extracting', f.name, f'({f.stat().st_size / 1e9:.2f} GB)')
        fmt = 'zip' if zipfile.is_zipfile(f) else 'tar'
        shutil.unpack_archive(f, EXTRACT / f.stem, format=fmt)
    elif f.suffix == '.xml':
        pass  # already usable

xml_files = list(RAW.rglob('*.xml'))
seq_dirs = [p for p in RAW.rglob('MVI_*') if p.is_dir()]
print('\nXML files:', len(xml_files))
print('Folders with XML:', sorted({str(p.parent) for p in xml_files}))
print('Sequence image folders (MVI_*):', len(seq_dirs))
print('Parents of image folders:', sorted({str(p.parent) for p in seq_dirs}))
print('Example frames:', sorted(p.name for p in seq_dirs[0].glob('*.jpg'))[:3] if seq_dirs else None)

## 5. Set the paths ⏸️

Your mentor will confirm these two paths from the output above.

In [ ]:
XML_DIR = '/content/raw/annotations'   # folder with the XML files (to confirm)
IMG_DIR = '/content/raw/extracted'     # folder that contains the MVI_* folders (to confirm)
OUT_DIR = '/content/data/ua_detrac'

## 6. Visual check of the source classes ⏸️

Is `others` really trucks? We also look at `van` to see what it looks like. Send both images to your mentor.

In [ ]:
!python scripts/sample_class_crops.py --xml-dir {XML_DIR} --images-dir {IMG_DIR} --label others --n 100 --out outputs/crops_others.jpg
!python scripts/sample_class_crops.py --xml-dir {XML_DIR} --images-dir {IMG_DIR} --label van --n 100 --out outputs/crops_van.jpg
from IPython.display import Image, display
display(Image('outputs/crops_others.jpg', width=900))
display(Image('outputs/crops_van.jpg', width=900))

## 7. Convert to YOLO (1 frame out of 10)

In [ ]:
!python scripts/convert_detrac.py --xml-dir {XML_DIR} --images-dir {IMG_DIR} --out-dir {OUT_DIR} --step 10

## 8. Statistics and visual check ⏸️

Send the statistics and the preview image to your mentor.

In [ ]:
!python scripts/dataset_stats.py --data-dir {OUT_DIR} --n-preview 20 --preview outputs/preview_ua_detrac.jpg
display(Image('outputs/preview_ua_detrac.jpg', width=1200))

## 9. Save the converted dataset to Google Drive

In [ ]:
!du -sh {OUT_DIR}
!cd /content/data && zip -q -r ua_detrac_step10.zip ua_detrac
!cp /content/data/ua_detrac_step10.zip {DRIVE_DIR}/
!ls -lh {DRIVE_DIR}